# Threadlight: Scripture in the conversation, not beside it

Threadlight is a restrained, Scripture-grounded presence for shared digital rooms. This notebook exercises the same provider-neutral orchestration endpoint used by the Discord adapter and the live web experience.

## The judged behavior loop

1. Normalize recent room context.
2. Run deterministic safety checks.
3. Discern whether to respond, clarify, remain silent, or escalate.
4. Retrieve an attributed passage only when appropriate.
5. Compose a restrained response grounded in that retrieved text.
6. Apply final safety handling and render the result in the originating channel.

In [ ]:
import json
import os
import urllib.error
import urllib.request

THREADLIGHT_URL = os.environ.get("THREADLIGHT_URL", "http://localhost:8787").rstrip("/")

In [ ]:
def get_json(path):
    with urllib.request.urlopen(f"{THREADLIGHT_URL}{path}", timeout=15) as response:
        return json.load(response)

def post_json(path, payload):
    request = urllib.request.Request(
        f"{THREADLIGHT_URL}{path}",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=45) as response:
        return json.load(response)

health = get_json("/api/health")
readiness = get_json("/api/readiness")
print(json.dumps({"health": health, "readiness": readiness}, indent=2))

In [ ]:
scenario = {
    "scenarioId": "grief",
    "messages": [
        {
            "id": "note-1",
            "author": {"id": "maya", "name": "Maya"},
            "content": "We got the call this morning. Grandma is gone.",
            "createdAt": "2026-07-24T20:41:00.000Z",
        },
        {
            "id": "note-2",
            "author": {"id": "eli", "name": "Eli"},
            "content": "I do not know what to say, but I am here with you.",
            "createdAt": "2026-07-24T20:43:00.000Z",
        },
    ],
    "prompt": "What might help this room hold space for grief?",
}

result = post_json("/api/demo/respond", scenario)
print(json.dumps(result, indent=2))

In [ ]:
summary = {
    "action": result["decision"]["action"],
    "risk_level": result["decision"]["riskLevel"],
    "passage_reference": result.get("reply", {}).get("passage", {}).get("reference"),
    "translation": result.get("reply", {}).get("passage", {}).get("translation"),
    "ai_provider": result["trace"]["aiProvider"],
    "scripture_provider": result["trace"]["scriptureProvider"],
    "steps": [step["status"] for step in result["trace"]["steps"]],
}
assert summary["action"] in {"respond", "clarify", "silent", "escalate"}
assert all(status in {"completed", "skipped", "failed"} for status in summary["steps"])
print(json.dumps(summary, indent=2))

## Reproducibility and safety

- The response trace names the providers that actually ran.
- Passage text is supplied by the configured Scripture provider, never invented by the composition model.
- Raw room messages are processed in memory and are not written to application logs.
- Urgent language is intercepted by deterministic safety logic before AI composition.
- The development adapters and competition credential readiness are reported separately.